# Detecting noisy monitors

This notebook shows how to use the WhyLabs Monitor Diagnoser to customize the diagnosis of a noisy monitor. It interacts with the diagnoser to get information on noisy and failing monitors, and to make selections about which monitor, segment and columns to diagnose.

## Install requirements

In [1]:
%pip install -e .


Obtaining file:///Volumes/Workspace/hack/smart-config
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for WhyLabs-Monitor-Diagnoser (pyproject.toml) ... done
  Created wheel for WhyLabs-Monitor-Diagnoser: filename=WhyLabs_Monitor_Diagnoser-0.0.1-0.editable-py3-none-any.whl size=3253 sha256=7b4cbfe8c7d43b46817562de75e01238943321354a771ca71eae6da224702c26
  Stored in directory: /private/var/folders/kg/k2sb6xms2650ty85vy98q5qr0000gn/T/pip-ephem-wheel-cache-mw1sol4x/wheels/3b/90/fd/b769d4b005362ce18dbd94fe781f74806d1a79ffbe447812d7
Successfully built WhyLabs-Monitor-Diagnoser
  Attempting uninstall: WhyLabs-Monitor-Diagnoser
    Found existing installation: WhyLabs-Monitor-Diagnoser 0.0.1
    Uninstalling WhyLabs-Monitor-Diagnoser-0.0.1:
      Successfully uninstalled Wh

## Setup whylabs API connection

First, set up the information to connect to WhyLabs. Update the org_id, dataset_id and api_key in the following before running it.


In [2]:
import getpass
from whylabs_toolkit.monitor.diagnoser.helpers.utils import env_setup

org_id = 'org-0'
dataset_id = 'model-0'
api_key = getpass.getpass()
api_endpoint = 'https://songbird.development.whylabsdev.com'

env_setup(
    org_id=org_id,
    dataset_id=dataset_id,
    api_key=api_key,
    whylabs_endpoint=api_endpoint
)

Then initialize the Monitor Diagnoser with the org_id and dataset_id.

In [3]:
from whylabs_toolkit.monitor.diagnoser.monitor_diagnoser import MonitorDiagnoser
diagnoser = MonitorDiagnoser(org_id, dataset_id)

# Running a customized diagnosis
## Get the recommended diagnostic interval

Get the dataset start/end time, granularity, and a recommended diagnostic interval for the dataset. The diagnoser will use this interval unless you override it by setting the `diagnostic_interval` property.

In [4]:
lineage, granularity, interval = diagnoser.choose_dataset_batches()
lineage, granularity, interval

(TimeRange(start=datetime.datetime(2020, 10, 8, 0, 0, tzinfo=datetime.timezone.utc), end=datetime.datetime(2024, 4, 15, 21, 0, tzinfo=datetime.timezone.utc)),
 <Granularity.daily: 'daily'>,
 '2024-03-16T00:00:00.000Z/2024-04-15T00:00:00.000Z')

## Get information on noisy and failing monitors

Get information on how many anomalies are detected by each monitor in the dataset. The results are ordered so that the monitors with the most anomalies per column are first (i.e. monitors which are firing on the many batches for certain columns). Beyond that, results with a higher average number of anomalies per column are considered noisier.

In [5]:
import pandas as pd
noisy_monitors = diagnoser.detect_noisy_monitors()
noisy_monitors_df = pd.DataFrame.from_records([m.dict() for m in noisy_monitors])
noisy_monitors_df

,monitor_id,analyzer_id,metric,column_count,segment_count,anomaly_count,max_anomaly_per_column,min_anomaly_per_column,avg_anomaly_per_column,action_count,action_targets
0,adorable-goldenrod-lion-9438,adorable-goldenrod-lion-9438-analyzer,frequent_items,2,1,31,30,1,15,0,[]
1,unsightly-orchid-gorilla-4971,unsightly-orchid-gorilla-4971-analyzer,frequent_items,3,1,33,30,1,11,0,[]
2,concerned-skyblue-penguin-6734,concerned-skyblue-penguin-6734-analyzer,frequent_items,3,1,32,30,1,10,0,[]
3,proud-seagreen-carabeef-65,proud-seagreen-carabeef-65-analyzer,histogram,1,1,28,28,28,28,0,[]
4,kind-cyan-kangaroo-1253,kind-cyan-kangaroo-1253-analyzer,histogram,1,1,28,28,28,28,0,[]
...,...,...,...,...,...,...,...,...,...,...,...
93,numerical-drift-monitor-60dfcc,numerical-drift-analyzer-60dfcc,histogram,1,1,2,2,2,2,2,"[email, slack]"
94,stormy-olive-butterfly-8693,stormy-olive-butterfly-8693-analyzer,histogram,1,1,2,2,2,2,0,[]
95,fine-magenta-nightingale-9708,fine-magenta-nightingale-9708-analyzer,unique_est_ratio,26,1,39,2,1,1,0,[]
96,None,eager-violet-newt-4599-analyzer,count_null_ratio,21,1,28,2,1,1,0,[]


Once you have run `detect_noisy_monitors`, you can retrieve the result at any time via the `noisy_monitors` property. You can also retrieve
 information about monitors with analysis failures using `failed_monitors`. 

In [6]:
failed_monitors_df = pd.DataFrame.from_records([n.dict() for n in diagnoser.failed_monitors])
failed_monitors_df

,monitor_id,analyzer_id,metric,failed_count,max_failed_per_column,min_failed_per_column,avg_failed_per_column,action_count,action_targets
0,energetic-black-cobra-7838,energetic-black-cobra-7838-analyzer,unique_est,56,28,28,28,1,[email]
1,None,expensive-tomato-moose-6522-analyzer,median,2191,28,7,27,0,[]
2,good-cornsilk-bear-9359,good-cornsilk-bear-9359-analyzer,count_null,2163,28,7,27,0,[]
3,elated-gray-baboon-4620,elated-gray-baboon-4620-analyzer,count_null_ratio,58,28,2,19,1,[email]
4,expensive-tomato-moose-6522,csw-analyzer-2,median,1190,28,7,15,0,[]
5,missing-values-ratio-monitor-v9uywi,missing-values-ratio-analyzer-v9uywi,count_null_ratio,2609,25,2,24,1,[email]
6,curious-lemonchiffon-rabbit-7000,curious-lemonchiffon-rabbit-7000-analyzer,frequent_items,15,15,15,15,1,[test-sort]
7,clear-azure-starling-8883,clear-azure-starling-8883-analyzer,frequent_items,15,15,15,15,1,[test-sort]
8,light-mintcream-rhinoceros-3655,light-mintcream-rhinoceros-3655-analyzer,frequent_items,70,15,1,8,0,[]
9,handsome-lemonchiffon-eel-4222,handsome-lemonchiffon-eel-4222-analyzer,frequent_items,17,15,2,8,0,[]


From this information, the diagnoser chooses the most noisy monitor that has notification actions to diagnose. This choice can be overridden by setting the `monitor_id_to_diagnose` property of the diagnoser to the desired monitor id. 

In [7]:
diagnoser.monitor_id_to_diagnose

'adorable-goldenrod-lion-9438'

We can get the monitor object from the diagnoser, to see its display name and any other useful information.

In [8]:
diagnoser.monitor_to_diagnose

Monitor(metadata=Metadata(version=1, schemaVersion=1, updatedTimestamp=1676498472577, author='system', description=None), id='adorable-goldenrod-lion-9438', displayName='wrong-drift-crowded-orchid-coyote-2773', tags=None, analyzerIds=['adorable-goldenrod-lion-9438-analyzer'], schedule=ImmediateSchedule(type='immediate'), disabled=None, severity=3, mode=DigestMode(type='DIGEST', filter=None, creationTimeOffset=None, datasetTimestampOffset='P7D', groupBy=None), actions=[])

We can similarly see the configuration of the analyzer that is being diagnosed.


In [9]:
diagnoser.analyzer_to_diagnose

Analyzer(metadata=Metadata(version=2, schemaVersion=1, updatedTimestamp=1713279603124, author='user_c9292ec40407f7b580f0a2c90745ebfba2b9e6ea81c848ef944d31e48a45f98', description=None), id='adorable-goldenrod-lion-9438-analyzer', displayName=None, tags=['featureSelection:all'], schedule=FixedCadenceSchedule(type='fixed', cadence=<Cadence.daily: 'daily'>, exclusionRanges=None), disabled=None, disableTargetRollup=None, targetMatrix=ColumnMatrix(segments=[], type=<TargetLevel.column: 'column'>, include=['*'], exclude=['issue_d', <ColumnGroups.group_output: 'group:output'>, 'url'], profileId=None), dataReadinessDuration=None, batchCoolDownPeriod=None, backfillGracePeriodDuration=None, config=DriftConfig(schemaVersion=None, params=None, metric=<ComplexMetrics.frequent_items: 'frequent_items'>, type=<AlgorithmType.drift: 'drift'>, algorithm='hellinger', threshold=0.7, minBatchSize=1, baseline=TrailingWindowBaseline(datasetId=None, inheritSegment=None, type=<BaselineType.TrailingWindow: 'Trail

## Get information on noisy and failing segments in the analyzer

Now we use the diagnoser to get information about noisy and failing segments in the analyzer, so we can choose a segment to diagnose. The results are sorted so the segment with the most anomalies for the selected monitor is first.

In [10]:
from whylabs_toolkit.monitor.diagnoser.helpers.utils import segment_as_readable_text

noisy_segments = diagnoser.detect_noisy_segments()
noisy_segments_df = pd.DataFrame.from_records([n.dict() for n in noisy_segments])
noisy_segments_df['segment'] = [segment_as_readable_text(n.segment.tags) for n in noisy_segments]
noisy_segments_df

,segment,total_anomalies,batch_count
0,overall,31,30


The diagnoser chooses the noisiest segment to diagnose. This can be changed by setting the `diagnostic_segment` property.

In [11]:
segment_as_readable_text(diagnoser.diagnostic_segment.tags)

'overall'

## Get information on noisy columns 

The next step is to get information on the noisy columns within the segment, so we can choose a subset of columns to diagnose. 

In [12]:
noisy_columns = diagnoser.detect_noisy_columns()
noisy_columns_df = pd.DataFrame.from_records([n.dict() for n in noisy_columns])
noisy_columns_df

,column,total_anomalies
0,issue_d,30
1,url,1
2,debt_settlement_flag,0
3,desc,0
4,disbursement_method,0
5,earliest_cr_line,0
6,emp_length,0
7,emp_title,0
8,grade,0
9,hardship_flag,0


The API limits diagnosis to 100 columns at a time, so we choose the top 100 noisy columns. We could then iterate through other columns if desired.

In [13]:
columns = list(noisy_columns_df.column[:100])
columns

['issue_d',
 'url',
 'debt_settlement_flag',
 'desc',
 'disbursement_method',
 'earliest_cr_line',
 'emp_length',
 'emp_title',
 'grade',
 'hardship_flag',
 'home_ownership',
 'id',
 'initial_list_status',
 'last_credit_pull_d',
 'last_pymnt_d',
 'loan_status',
 'next_pymnt_d',
 'purpose',
 'pymnt_plan',
 'sub_grade',
 'term',
 'title',
 'verification_status',
 'verification_status_joint',
 'addr_state',
 'zip_code',
 'application_type']

## Ask for a monitor diagnosis


In [17]:
# for now, we need to enforce this to run using local server
import os
os.environ['USE_LOCAL_SERVER'] = 'server'
monitor_report = diagnoser.diagnose(columns)

In [18]:
print(monitor_report.describe())

Diagnosis is for monitor "wrong-drift-crowded-orchid-coyote-2773" [adorable-goldenrod-lion-9438] in model-0 org-0, over interval 2024-03-16T00:00:00.000Z/2024-04-15T00:00:00.000Z.

Analyzer is drift configuration for frequent_items metric with TrailingWindow baseline.
Analyzer "adorable-goldenrod-lion-9438-analyzer" targets 123 columns and ran on 27 columns in the diagnosed segment.


Diagnostic segment is "overall".
Diagnostic interval contains 30 batches.

Diagnostic interval rollup contains 2494691 rows for the diagnosed columns.

Analysis results summary:
Found non-failed results for 27 columns and 30 batches.
Found 31 anomalies in 2 columns, with up to 100.0% (30) batches having anomalies per column and 50.0% (15.0) on average.
Columns with anomalies are:
|         |   0 |
|:--------|----:|
| issue_d |  30 |
| url     |   1 |

No failures were detected.

Conditions that may impact diagnosis quality include:
	* analyzer_changed: Analyzer changed within the diagnostic interval - det